# 03 - Model Training and Evaluation

This notebook reviews the reproducible training output and prioritizes ranking quality using PR-AUC, ROC-AUC, top-decile lift, and cumulative capture.

In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT / "src"))

## Candidate Model Comparison

In [2]:
import pandas as pd

model_comparison = pd.read_csv(PROJECT_ROOT / "outputs" / "reports" / "model_comparison.csv")
model_comparison.sort_values("pr_auc", ascending=False)

,model_name,roc_auc,pr_auc,precision,recall,f1,balanced_accuracy,top_decile_lift,capture_rate_top_10_pct,capture_rate_top_20_pct,capture_rate_top_30_pct,training_seconds
0,Hist Gradient Boosting,0.858452,0.368458,0.284269,0.932134,0.435672,0.802154,3.224881,0.322522,0.582638,0.794048,7.76
1,Random Forest,0.857312,0.366787,0.285191,0.925391,0.436011,0.800704,3.218459,0.321880,0.579319,0.790409,28.62
2,Logistic Regression,0.849356,0.339739,0.286051,0.910512,0.435334,0.796539,2.959441,0.295975,0.558981,0.780026,25.18


## Validation Metrics

In [3]:
import json

metrics = json.loads((PROJECT_ROOT / "outputs" / "reports" / "metrics.json").read_text(encoding="utf-8"))
pd.Series(
    {
        "roc_auc": metrics["roc_auc"],
        "pr_auc": metrics["pr_auc"],
        "top_decile_lift": metrics["top_decile_lift"],
        "capture_rate_top_30_pct": metrics["capture_rate_top_30_pct"],
        "best_threshold": metrics["best_threshold"]["threshold"],
    }
).to_frame("value")

,value
roc_auc,0.858452
pr_auc,0.368458
top_decile_lift,3.224881
capture_rate_top_30_pct,0.794048
best_threshold,0.650000


## Decile Targeting Performance

In [4]:
validation_deciles = pd.read_csv(PROJECT_ROOT / "outputs" / "reports" / "validation_decile_report.csv")
validation_deciles.head(10)

,propensity_decile,customers,responders,avg_score,min_score,max_score,response_rate,lift,capture_rate,cumulative_customers,cumulative_responders,cumulative_capture_rate,population_share
0,1,7623,3013,0.813093,0.787776,0.890795,0.395251,3.224881,0.322522,7623,3013,0.322522,0.100010
1,2,7622,2430,0.765451,0.743720,0.787776,0.318814,2.601224,0.260116,15245,5443,0.582638,0.200008
2,3,7622,1975,0.708483,0.660197,0.743705,0.259118,2.114164,0.211411,22867,7418,0.794048,0.300005
3,4,7622,1271,0.591545,0.503343,0.660197,0.166754,1.360558,0.136052,30489,8689,0.930101,0.400003
4,5,7622,560,0.357419,0.191684,0.503330,0.073472,0.599459,0.059944,38111,9249,0.990045,0.500000
5,6,7622,83,0.078772,0.006702,0.191638,0.010890,0.088848,0.008885,45733,9332,0.998930,0.599997
6,7,7622,7,0.004401,0.003275,0.006702,0.000918,0.007493,0.000749,53355,9339,0.999679,0.699995
7,8,7622,1,0.003263,0.003182,0.003275,0.000131,0.001070,0.000107,60977,9340,0.999786,0.799992
8,9,7622,0,0.002857,0.002429,0.003182,0.000000,0.000000,0.000000,68599,9340,0.999786,0.899990
9,10,7623,2,0.002293,0.002213,0.002429,0.000262,0.002141,0.000214,76222,9342,1.000000,1.000000


## Model Diagnostic Chart Inventory

In [5]:
chart_dir = PROJECT_ROOT / "outputs" / "charts"
sorted(path.name for path in chart_dir.glob("*.png"))

['annual_premium_distribution.png',
 'calibration_curve.png',
 'confusion_matrix.png',
 'dashboard_executive_preview.png',
 'feature_importance.png',
 'gain_chart.png',
 'lift_chart.png',
 'precision_recall_curve.png',
 'response_distribution.png',
 'roc_curve.png',
 'segment_response_rate.png']